# Lab 1 — Predicting Post Performance Relative to the Author

**Final selected solution:** Task-adapted embedding + SVD probability ensemble.  
**Mean author-split validation macro-F1:** 0.6129.  
**Follow-up official-test macro-F1:** 0.6008.  
**Prediction file:** `results/predictions/lab1_final_v4.csv`.

This is the final English report. Section 4 records the earlier frozen-encoder ensemble comparison; Section 9 describes the subsequent task-specific encoder adaptation and final selection. This submission includes the final method, aggregate results and the required prediction CSV. Personal learning notebooks, source dataset records and trained checkpoints are excluded. Predictions can be regenerated with authorized dataset access.

**Evaluation disclosure:** test outcomes were inspected during earlier rounds, and the search was reopened afterward. Final configurations were selected on validation before their follow-up test assessments. This is **not an untouched blind test**. Macro-F1 is not the percentage of correct predictions.


## 1. Task and Data

One object is an author's own post. The target is `below` (0), `typical` (1), or `above` (2), relative to the author's usual performance. The released labels are used unchanged. The official training set contains 71,395 labelled posts from 3,295 authors; the official test contains 17,623 labelled posts from 824 different authors.

The historical level is the median of `log(1 + reactions)` for strictly earlier own posts. At least five historical posts are required for a level. The current post's reactions, residual and target label are never model input features. Plain reposts are excluded. Images and videos are not processed.


In [1]:
from pathlib import Path
import json
import pandas as pd

# Run this notebook from submissions/wilsonSev (the submission directory).
import os

DATA_DIR = Path(os.environ.get("KAIROS_DATA_DIR", "data"))
round3 = Path("results/round3")
validation = pd.DataFrame(json.loads((round3 / "validation_results.json").read_text()))
selection = json.loads(Path("results/adaptation/selection.json").read_text())
assessment = json.loads(Path("results/adaptation/test_results.json").read_text())
method_names = {
    "embeddings_balanced": "Embedding hybrid, balanced classes",
    "svd_balanced": "SVD hybrid, balanced classes",
    "equal_weight_ensemble": "Equal-weight probability ensemble",
    "embeddings_natural": "Embedding hybrid, natural class frequencies",
}


## 2. Evaluation Protocol

Each validation split holds out 20% of the **authors** from the official training set. The fixed seeds are 42, 7 and 2026. All posts by a given author stay on one side. Classifier training, age imputation, TF-IDF vocabulary, IDF and SVD fitting use only the fit portion. The earlier frozen encoder is not trained on this dataset; the final adapted encoder is trained only on its split's fit posts (Section 9). Historical features for a validation author use that author's strictly earlier posts, including the initial unlabelled history, without using validation labels.

The three splits overlap and were reused for model selection; they are sensitivity checks rather than three independent tests. The final choice maximizes their mean **unrounded** macro-F1. The final ensemble weight is fixed at 50/50. There is no blend-weight or decision-threshold grid search. The primary-split ensemble result was seen before the last bounded comparison; this adaptive selection is disclosed.

Macro-F1 gives equal weight to the F1 of each class. QWK, accuracy and per-class recall/F1 are secondary diagnostics. Natural class weighting can improve accuracy while reducing macro-F1.

The provided histogram gradient booster is retained: at most 300 boosting iterations, learning rate 0.08, random seed 42 and scikit-learn's inherited default early stopping. Its internal early-stopping subset is row-based; the outer assessment remains author-disjoint.


## 3. Features and Models

**Shared tabular features.** Sixteen original numeric features describe content length, punctuation/links, posting time, historical author level/spread, followers, post age and history count. Three added style features count exclamation marks, measure the uppercase-letter ratio and the average word length.

**BM25 history.** For each post, retrieve up to three positive-score matches among the same author's strictly earlier posts (`k1=1.2`, `b=0.75`). Use their mean log-reaction count minus the author's historical median, plus the number of matches. No-match and neutral-result cases are therefore distinguishable. Same-timestamp posts do not enter one another's history.

**Recent history.** Four features describe the last three and ten posts relative to the full past median, the last post's deviation, and the population standard deviation of the last three log-reaction counts. Short histories use available posts; empty history uses zero.

**SVD branch.** TF-IDF (`min_df=5`, at most 20,000 terms), followed by 128-dimensional TruncatedSVD (`n_iter=7`, seed 42) and L2 normalization. Vocabulary, IDF and reduction are refitted for every fit split. These dimensions are added to the shared tabular features.

**Embedding branch.** The initial frozen model was `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, pinned revision `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`. It produces normalized 384-dimensional vectors; empty texts use zero vectors. Its default 128-token limit truncates long posts. These dimensions are added to the shared tabular features.

Each branch uses StandardScaler for numeric columns, OneHotEncoder for content type (`handle_unknown='ignore'`, `min_frequency=20`), and the same histogram gradient boosting classifier. Author/post identifiers and labels are excluded from the feature matrix. Embedding coordinates are numerical features, not individually named topics.

**Final adaptation:** Section 9 replaces frozen encoder weights with fit-only task-adapted weights. The underlying architecture, dimensionality, input limit and history features are unchanged.


## 4. Earlier Comparison: Frozen Encoders and Class Weighting

**Hypothesis A — complementary representations.** The SVD and embedding hybrids may make different errors. Average their probability vectors and select the class with the largest mean probability. This is late fusion / soft voting. Probabilities are aligned to classes `[0, 1, 2]` and exactly the same post order. Individual model probability estimates are not assumed to be perfectly calibrated; empirical validation determines whether their fixed mean helps.

**Hypothesis B — class weighting.** Repeat the embedding hybrid with `class_weight=None` instead of `balanced`, leaving all other settings unchanged. This is a training-configuration ablation, not a new architecture. It tests whether the original class weighting remains suitable after richer features are added.

Only four candidates are compared: both balanced hybrids, their equal-weight ensemble, and the embedding hybrid with natural class frequencies. No new large encoder is downloaded or tested in this round.

The following tables document the earlier round. The subsequent encoder-adaptation comparison and latest selection are in Section 9.


In [2]:
validation_table = validation.pivot(
    index="experiment", columns="seed", values="macro_f1_unrounded"
)
validation_table = validation_table.reindex(columns=[42, 7, 2026])
validation_table["Mean macro-F1"] = validation_table.mean(axis=1)
validation_table.index = validation_table.index.map(method_names)
validation_table.sort_values("Mean macro-F1", ascending=False).round(4)


seed,42,7,2026,Mean macro-F1
experiment,,,,
Equal-weight probability ensemble,0.6096,0.6097,0.6101,0.6098
"Embedding hybrid, balanced classes",0.6066,0.6057,0.6030,0.6051
"SVD hybrid, balanced classes",0.6049,0.5977,0.6053,0.6026
"Embedding hybrid, natural class frequencies",0.6006,0.5900,0.5958,0.5954


In [3]:
secondary = validation.groupby("experiment")[
    ["macro_f1_unrounded", "accuracy", "qwk"]
].mean()
secondary.index = secondary.index.map(method_names)
secondary.rename(columns={"macro_f1_unrounded": "macro_f1"}).sort_values(
    "macro_f1", ascending=False
).round(4)


,macro_f1,accuracy,qwk
experiment,,,
Equal-weight probability ensemble,0.6098,0.6187,0.5550
"Embedding hybrid, balanced classes",0.6051,0.6145,0.5488
"SVD hybrid, balanced classes",0.6026,0.6109,0.5462
"Embedding hybrid, natural class frequencies",0.5954,0.6926,0.5031


## 5. Final Selection and Follow-Up Test

After the earlier frozen-encoder comparison in Section 4, task-specific adaptation was tested as described in Section 9. The final rule selected **Task-adapted embedding + SVD probability ensemble**, mean validation macro-F1 **0.6129**. This decision was written to `results/adaptation/selection.json` before the final full-train adaptation and follow-up test assessment.

A fresh encoder was adapted using only sampled official-train posts. The classifier then used all 71,395 train posts; the SVD companion was also refitted on train when selected. No test labels were used for encoder adaptation, feature fitting, model fitting or selection among candidates.

The test had been inspected in earlier rounds, so the score is a follow-up assessment, not an unbiased blind-test estimate. The validation-selected configuration remains final regardless of its test score.


In [4]:
test_comparison = pd.DataFrame(
    [
        {
            "Model": "Provided baseline",
            "macro_f1": 0.5410,
            "qwk": 0.4576,
            "accuracy": 0.5465,
        },
        {
            "Model": "First-round BM25 hybrid",
            "macro_f1": 0.5633,
            "qwk": 0.4950,
            "accuracy": 0.5668,
        },
        {
            "Model": "Frozen embedding hybrid",
            "macro_f1": 0.5953,
            "qwk": 0.5365,
            "accuracy": 0.6027,
        },
        {
            "Model": "Frozen-encoder ensemble",
            "macro_f1": 0.5995,
            "qwk": 0.5399,
            "accuracy": 0.6073,
        },
        {"Model": "Final task-adapted solution", **assessment["results"]},
    ]
)
test_comparison


,Model,macro_f1,qwk,accuracy
0,Provided baseline,0.5410,0.4576,0.5465
1,First-round BM25 hybrid,0.5633,0.4950,0.5668
2,Frozen embedding hybrid,0.5953,0.5365,0.6027
3,Frozen-encoder ensemble,0.5995,0.5399,0.6073
4,Final task-adapted solution,0.6008,0.5433,0.6084


In [5]:
pd.DataFrame(assessment["per_class"]).round(4)


,class,precision,recall,f1,support
0,below,0.4328,0.6798,0.5289,3541
1,typical,0.7599,0.5453,0.6350,10491
2,above,0.5722,0.7224,0.6386,3591


In [6]:
pd.DataFrame(
    assessment["confusion_rows_true"],
    index=["Actual below", "Actual typical", "Actual above"],
    columns=["Predicted below", "Predicted typical", "Predicted above"],
)


,Predicted below,Predicted typical,Predicted above
Actual below,2407,1001,133
Actual typical,2964,5721,1806
Actual above,190,807,2594


## 6. Interpretation and Limits

The experiments support combining textual representations with author history rather than relying on text alone. Any benefit of the ensemble is an observed predictive improvement on these splits; it does not establish a causal effect of particular words or writing styles on engagement. The additional class-weight experiment is reported even when it does not win.

Limitations include repeated/adaptive use of validation, previous exposure to official test scores, the encoder's 128-token truncation, and no image/video features. Historical counters are crawl-time snapshots and may not have been available at those values at the original posting time. Followers and crawl-age features are benchmark artifacts and may differ from information available in a live pre-publication product. Strict timestamp exclusion prevents current/future-post leakage but does not resolve snapshot-availability limitations.

Feature-history checks cover hand calculations, equal timestamps, missing dates, author separation, and invariance to current/future reaction changes. Final predictions are checked for exact test-key coverage, unique keys and valid labels.


## 7. Reproduction and Deliverables

Saved results can be inspected without the dataset. For full reproduction use Python 3.12 with `requirements-round2.txt`, supply the unpacked core dataset, and set `REBUILD_FINAL=True`. The encoder runs locally; no post text is sent to a remote inference service. Frozen and adapted model weights, dataset files and embedding arrays are excluded from the archive.

The reproduction cell rebuilds the frozen controls, fits separate adapted encoders for each author split, freezes the validation selection, then adapts a fresh full-train encoder and evaluates the selected solution. The earlier official-test exposure is part of the documented experimental history; this does not create a blind test. Display cells were executed and checked, but a second complete training run from a clean machine has not been performed.


In [7]:
REBUILD_FINAL = False
if REBUILD_FINAL:
    import os
    import subprocess
    import sys

    environment = dict(os.environ, KAIROS_DATA_DIR=str(DATA_DIR))
    Path("results").mkdir(exist_ok=True)
    encoder_python = Path(".venv-embeddings/bin/python")
    encoder_python = (
        str(encoder_python.absolute()) if encoder_python.exists() else sys.executable
    )

    def run_with(executable, *arguments):
        subprocess.run([executable, *arguments], env=environment, check=True)

    recent_cache = Path("results/fusion/train_recent_table.pkl")
    if recent_cache.exists():
        recent_cache.unlink()
    run_with(sys.executable, "prepare_fusion_data.py")
    run_with(encoder_python, "encode_posts.py")
    run_with(
        sys.executable, "lab_round3.py", "--stage", "validation", "--rebuild-primary"
    )
    run_with(sys.executable, "lab_round3.py", "--stage", "final")
    for seed in [42, 7, 2026]:
        run_with(
            sys.executable,
            "adaptation_pilot.py",
            "--stage",
            "prepare",
            "--seed",
            str(seed),
        )
        run_with(encoder_python, "adapt_text_encoder.py", "--seed", str(seed))
        run_with(
            sys.executable,
            "adaptation_pilot.py",
            "--stage",
            "assess",
            "--seed",
            str(seed),
        )
    run_with(sys.executable, "final_adaptation.py", "--stage", "select")
    latest_selection = json.loads(Path("results/adaptation/selection.json").read_text())
    if latest_selection["selected"].startswith("adapted_"):
        run_with(sys.executable, "adaptation_pilot.py", "--stage", "prepare", "--full")
        run_with(encoder_python, "adapt_text_encoder.py", "--full")
        run_with(sys.executable, "final_adaptation.py", "--stage", "final")
else:
    print("Displaying saved verified results. Set REBUILD_FINAL=True to retrain.")


Displaying saved verified results. Set REBUILD_FINAL=True to retrain.


In [8]:
prediction_path = Path("results/predictions/lab1_final_v4.csv")
if prediction_path.exists():
    predictions = pd.read_csv(prediction_path)
    assert len(predictions) == assessment["n_test"]
    assert list(predictions.columns) == ["author", "post", "label"]
    assert not predictions.duplicated(["author", "post"]).any()
    assert predictions["label"].isin([0, 1, 2]).all()
    print(
        f"Validated {len(predictions):,} prediction rows; record identifiers are not displayed."
    )
else:
    print("Prediction file not found. Retrain with authorized data to regenerate it.")


Validated 17,623 prediction rows; record identifiers are not displayed.


## 8. Connection to the Course

- Lecture 2, slides 21 and 31–33: compact text representations, embeddings, normalization and a classifier on frozen representations.
- Lecture 2, slide 37: early fusion and late fusion. Each hybrid uses early fusion; the final averaging experiment uses late fusion.
- Lecture 2, slide 49: changing author profiles and recent history.
- Lecture 3 Part 1, slide 45: retrieve an author's earlier posts and use their outcomes as features.
- [Sentence Transformers model card](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2): encoder architecture, dimensionality and input limit.
- [scikit-learn voting documentation](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier): averaging class probabilities.

The lecture concepts motivate experiments; they do not guarantee a particular macro-F1 score.


## 9. Task-Specific Encoder Adaptation: Final Extension

**Hypothesis:** general semantic similarity is not the same as relevance to author-relative post performance. Adapt the encoder to the training labels, then combine its representations with the unchanged author-history/BM25 features.

This is **SetFit-inspired pairwise adaptation**, implemented with Sentence Transformers and PyTorch, not a run of the SetFit package or its default logistic-regression head. The classifier here remains the history-aware balanced histogram gradient booster. This is the first experiment in which the text encoder weights themselves are updated.

For each author split, start again from the same pinned pretrained encoder. Sample 128 non-empty distinct fit texts per class (384 posts), form one different same-class partner and one different-class partner per anchor, and shuffle the resulting 768 pairs. Optimize MSE between cosine similarity and targets 1 (same class) / 0 (different class). Fixed settings: AdamW learning rate 0.00002, weight decay 0.01, gradient norm clipping 1, pair batch size 16, one epoch / 48 updates, random seed 1234, maximum 128 tokens. All encoder parameters are trainable. The limited sample and short schedule were chosen to bound runtime, not by searching validation scores.

Validation authors and labels are excluded from pair construction and encoder adaptation. Pair membership and targets were checked against the fit table. Encoder weights were verified to change, and generated vectors were checked for finite values and unit length (or zero for empty text). A separate encoder is trained for each split; a model adapted on one split is never reused for another split's validation.

The seed-42 pilot passed the predeclared continuation gate of at least +0.002 macro-F1 over the incumbent ensemble. The same fixed training settings were then checked on seeds 7 and 2026. These are reused, overlapping validation splits; the search remains adaptive. The final candidates are the incumbent ensemble, the adapted embedding hybrid and its fixed 50/50 probability ensemble with the original SVD hybrid.

| Candidate | Seed 42 | Seed 7 | Seed 2026 | Mean macro-F1 |
|---|---:|---:|---:|---:|
| Frozen-encoder ensemble (previous final) | 0.6096 | 0.6097 | 0.6101 | 0.6098 |
| Adapted embedding hybrid | 0.6116 | 0.6043 | 0.6043 | 0.6067 |
| Adapted embedding + SVD ensemble | 0.6167 | 0.6099 | 0.6120 | 0.6129 |

These are point estimates from three overlapping author splits, not a statistical significance test. In particular, the gain on seed 7 is very small; the results do not support a claim of a large, guaranteed improvement.

Selected by highest mean unrounded validation macro-F1: **Task-adapted embedding + SVD probability ensemble** (**0.6129**). For final fitting, a fresh encoder is adapted on 384 sampled posts from the full official training set, then the classifier uses all 71,395 training posts. The SVD companion, when selected, is also refitted on full train. The official test is assessed only for this selected configuration; it was already seen in earlier rounds, so this is not a blind test.

Final follow-up official-test macro-F1: **0.6008**. The final deliverable is **`results/predictions/lab1_final_v4.csv`**. Selection remains validation-based regardless of the follow-up test result.

The results support this particular bounded adaptation recipe; they do not establish that larger models or more training always help. Performance labels are author-dependent and need not imply semantic similarity, so the history features remain important. The 128-token truncation, snapshot counters and adaptive evaluation limitations still apply.

Implementation: `adaptation_pilot.py` prepares fit-only pairs and evaluates candidates; `adapt_text_encoder.py` performs local adaptation and encoding; `final_adaptation.py` freezes the selection and fits the final classifier(s). Model checkpoints and text caches remain local and are not in the submission archive.

Course connection: Lecture 2, slides 32–33 (SetFit and task-adapted embeddings), alongside the early/late-fusion concepts already used. Reference: [official SetFit conceptual guide](https://huggingface.co/docs/setfit/en/conceptual_guides/setfit).
